# Excited states solvers

**注意**: 此版本已更新为适配 Qiskit 1.x

## Introduction

在本教程中，我们将讨论 Qiskit Nature 的激发态计算接口。目标是计算分子哈密顿量的激发态。

In [ ]:
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver

driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 1.4",
    basis="sto3g",
    charge=0,
    spin=0,
    unit=DistanceUnit.ANGSTROM,
)

es_problem = driver.run()

使用 Jordan-Wigner 映射

In [ ]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()

## The Solver

首先使用 `NumPyEigensolver` 进行经典精确对角化计算。

In [ ]:
from qiskit_algorithms import NumPyEigensolver

numpy_solver = NumPyEigensolver(k=4, filter_criterion=es_problem.get_default_filter_criterion())

## The calculation and results

计算并输出 NumPyEigensolver 的结果。

In [ ]:
from qiskit_nature.second_q.algorithms import ExcitedStatesEigensolver

numpy_excited_states_solver = ExcitedStatesEigensolver(mapper, numpy_solver)
numpy_results = numpy_excited_states_solver.solve(es_problem)

print("=== NumPy 精确对角化结果 ===")
print(numpy_results)

## 使用自定义 filter 函数

只过滤粒子数和磁化强度，获取所有自旋态。

In [ ]:
import numpy as np

def filter_criterion(eigenstate, eigenvalue, aux_values):
    return np.isclose(aux_values["ParticleNumber"][0], 2.0) and np.isclose(
        aux_values["Magnetization"][0], 0.0
    )

new_numpy_solver = NumPyEigensolver(k=4, filter_criterion=filter_criterion)
new_numpy_excited_states_solver = ExcitedStatesEigensolver(mapper, new_numpy_solver)
new_numpy_results = new_numpy_excited_states_solver.solve(es_problem)

print("=== 使用自定义 filter 的 NumPy 结果 ===")
print(new_numpy_results)

## VQE 基态计算 (可选)

使用 VQE 计算基态能量。

In [ ]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SLSQP
from qiskit.primitives import Estimator
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
import warnings

# 忽略弃用警告
warnings.filterwarnings('ignore', category=DeprecationWarning)

ansatz = UCCSD(
    es_problem.num_spatial_orbitals,
    es_problem.num_particles,
    mapper,
    initial_state=HartreeFock(
        es_problem.num_spatial_orbitals,
        es_problem.num_particles,
        mapper,
    ),
)

estimator = Estimator()
solver = VQE(estimator, ansatz, SLSQP())
solver.initial_point = [0.0] * ansatz.num_parameters
gse = GroundStateEigensolver(mapper, solver)

try:
    vqe_results = gse.solve(es_problem)
    print("=== VQE 基态计算结果 ===")
    print(vqe_results)
except Exception as e:
    print(f"VQE 计算遇到问题：{e}")
    print("这可能是因为数值精度问题，不影响 NumPy 结果的准确性")

## 版本信息

In [ ]:
import qiskit
import qiskit_nature
import qiskit_algorithms

print(f"Qiskit version: {qiskit.__version__}")
print(f"Qiskit Nature version: {qiskit_nature.__version__}")
print(f"Qiskit Algorithms version: {qiskit_algorithms.__version__}")